<a href="https://colab.research.google.com/github/jdmartinev/CVBootcampMCDA/blob/main/notebooks/03_competition_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Workshop CLIP — Notebook 03: Competition

---

### La tarea

Se te dan **3 imágenes de referencia**. Para cada una debes:

1. **Inventarte un query de texto** que la describa — no puedes usar captions del dataset
2. Usar ese query para **recuperar las top-10 imágenes** más similares del corpus
3. Opcionalmente combinar la señal de texto con la señal visual de la imagen de referencia

**Métrica — Overlap@10:** tus top-10 se comparan contra el top-10 que recuperaría un oracle que conoce los captions reales. Cuanto mejor describes la imagen con tu query, más se solapan los resultados.

$$\text{score} = \frac{1}{3} \sum_{i=1}^{3} \frac{|\text{top-10}_{\text{tuyo}}(i) \cap \text{top-10}_{\text{oracle}}(i)|}{10}$$

**Lo que diferencia a los equipos:** la calidad del query inventado + el método de fusión texto-imagen.

---

**TODOs obligatorios:** A, B  
**TODO extra:** C (fusión avanzada)  
**Corpus:** 2000 imágenes | **Límite:** 3 submissions/día

> ⚠️ La indexación de 2000 imágenes tarda ~5 min en CPU. Ejecútala una sola vez.

---
## 0 — Setup

In [ ]:
%%capture
!pip install transformers datasets Pillow ipywidgets pandas

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
clip_model.eval()
print("✅ Modelo cargado")

### 0.3 — Pega aquí tus funciones de los notebooks anteriores

In [ ]:
# ── NB01 ──────────────────────────────────────────────────────────────────────
def get_text_embeddings(texts, model, processor, device):
    raise NotImplementedError("Copia tu implementación del NB01")

def get_image_embeddings(images, model, processor, device):
    raise NotImplementedError("Copia tu implementación del NB01")

def compute_scores(query_emb, corpus_emb):
    raise NotImplementedError("Copia tu implementación del NB01")

# ── NB02 ──────────────────────────────────────────────────────────────────────
def build_image_index(images, model, processor, device, batch_size=32):
    raise NotImplementedError("Copia tu implementación del NB02")

def search_by_text(query_text, image_index, model, processor, device, top_k=10):
    raise NotImplementedError("Copia tu implementación del NB02")

---
## 1 — Corpus (2000 imágenes)

In [ ]:
CORPUS_SIZE = 2000
SEED = 42

print("Cargando Flickr30k...")
raw = load_dataset("Mozilla/flickr30k-transformed-captions", split="test")

random.seed(SEED)
all_indices = list(range(len(raw)))
random.shuffle(all_indices)
corpus_indices = all_indices[:CORPUS_SIZE]

corpus_images   = [raw[i]["image"].convert("RGB") for i in tqdm(corpus_indices, desc="Cargando")]
corpus_captions = [raw[i]["original_alt_text"] for i in corpus_indices]
corpus_ids      = [str(raw[i]["img_id"]) for i in corpus_indices]

print(f"✅ {len(corpus_images)} imágenes | {len(set(corpus_ids))} IDs únicos")

In [ ]:
# ~5 min en CPU — ejecutar una sola vez
image_index = build_image_index(corpus_images, clip_model, clip_processor, device)
print(f"✅ Índice: {image_index.shape}")

---
## 2 — Imágenes de referencia

Estas son las **3 imágenes fijas** para todos los equipos. Tu objetivo es describirlas con texto mejor que los demás.

In [ ]:
REFERENCE_INDICES = [2, 8, 19]
REFERENCE_IDS     = ["ref_0", "ref_1", "ref_2"]

reference_images = [corpus_images[i] for i in REFERENCE_INDICES]
reference_embs   = get_image_embeddings(reference_images, clip_model, clip_processor, device)
# reference_embs shape: (3, 512)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, rid in zip(axes, reference_images, REFERENCE_IDS):
    ax.imshow(img)
    ax.set_title(f"Imagen de referencia: {rid}", fontsize=11, fontweight="bold")
    ax.axis("off")
plt.suptitle("Describe cada imagen con un query de texto — sin usar captions del dataset",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## 3 — TODO A: Define tus queries

Escribe un query de texto para cada imagen de referencia. **Reglas:**

- No puedes copiar captions del dataset
- Debe ser una descripción en inglés
- Puedes ser tan específico o general como quieras — esa es la decisión clave

> 💡 Piensa en qué aspectos de la imagen son más discriminativos: ¿el sujeto, la acción, el entorno, los colores, el ambiente? Un query como `"sport"` es demasiado genérico. Uno como `"soccer player in white jersey kicking ball on green field with crowd"` es mucho más informativo.

In [ ]:
# TODO A — Define tus queries para cada imagen de referencia
# Modifica los strings. Sé descriptivo.

my_queries = {
    "ref_0": "???",   # imagen de fútbol
    "ref_1": "???",   # imagen de BMX
    "ref_2": "???",   # imagen de club nocturno
}

# Verificación — no modifiques esto
assert all(q != "???" for q in my_queries.values()), \
    "❌ Reemplaza los '???' con tus queries"
assert all(len(q) > 10 for q in my_queries.values()), \
    "❌ Los queries son demasiado cortos — sé más descriptivo"
print("✅ Queries definidos:")
for rid, q in my_queries.items():
    print(f"  {rid}: '{q}'")

---
## 4 — TODO B: Función de búsqueda por referencia

Implementa `search_by_reference` — la función central del reto. Dado un query de texto **y** el embedding de la imagen de referencia, debe retornar las top-k imágenes del corpus.

La forma más simple es la **fusión lineal**:

$$\text{score}(i) = \alpha \cdot \text{sim}_{\text{texto}}(i) + (1 - \alpha) \cdot \text{sim}_{\text{imagen}}(i)$$

Pero puedes implementar lo que quieras — esa es la parte creativa del reto.

**Parámetros que tienes para jugar:**
- `alpha` — balance texto vs imagen
- El query de texto en sí (TODO A)
- El método de fusión (lineal, RRF, re-ranking, ...)

In [ ]:
def search_by_reference(
    query_text: str,
    reference_emb: torch.Tensor,
    image_index: torch.Tensor,
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
    top_k: int = 10,
    alpha: float = 0.7,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Recupera las top_k imágenes combinando un query de texto
    con el embedding de una imagen de referencia.

    Args:
        query_text:    tu descripción de la imagen de referencia
        reference_emb: embedding de la imagen de referencia, shape (1, 512)
        alpha:         peso del texto vs imagen — puedes tunearlo por referencia

    Returns:
        (indices, scores) — Tensors de shape (top_k,)
    """
    # TODO B — implementar aquí
    # Mínimo: fusión lineal de scores de texto e imagen
    # Puedes ir más allá: RRF, múltiples queries, re-ranking, ...
    raise NotImplementedError


# ── Prueba visual (descomenta cuando TODO B esté listo) ────────────────────────
# for ref_id, ref_img, ref_emb, query in zip(
#     REFERENCE_IDS, reference_images,
#     reference_embs,         # iterar sobre las 3 filas
#     my_queries.values()
# ):
#     idx, scores = search_by_reference(
#         query, ref_emb.unsqueeze(0), image_index,
#         clip_model, clip_processor, device, top_k=5
#     )
#     fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
#     axes[0].imshow(ref_img)
#     axes[0].set_title(f"REF: {ref_id}\n'{query[:35]}...'", fontsize=7,
#                       color="crimson", fontweight="bold")
#     axes[0].axis("off")
#     for r, (i, s) in enumerate(zip(idx.tolist(), scores.tolist())):
#         axes[r+1].imshow(corpus_images[i])
#         axes[r+1].set_title(f"#{r+1} sim={s:.3f}", fontsize=8)
#         axes[r+1].axis("off")
#     plt.tight_layout(); plt.show()

---
## 5 — Evaluación local

Esta celda simula la métrica del leaderboard para que puedas iterar antes de hacer submit.

**Overlap@10:** fracción de tus top-10 que coincide con el top-10 del oracle (que usa los captions reales de Flickr30k).

In [ ]:
def compute_oracle_top10(
    reference_idx: int,
    image_index: torch.Tensor,
    corpus_captions: list,
    corpus_ids: list,
    k: int = 10,
) -> set:
    """
    Recupera el top-k usando los 5 captions reales de la imagen de referencia.
    La unión de los resultados es el set correcto.
    """
    real_captions = corpus_captions[reference_idx]  # lista de 5 captions
    oracle_ids = set()
    for caption in real_captions:
        indices, _ = search_by_text(
            caption, image_index,
            clip_model, clip_processor, device, top_k=k
        )
        for i in indices.tolist():
            oracle_ids.add(corpus_ids[i])
    return oracle_ids


def evaluate_local(
    my_queries: dict,
    reference_indices: list,
    reference_embs: torch.Tensor,
    image_index: torch.Tensor,
    corpus_ids: list,
    corpus_captions: list,
    alpha: float = 0.7,
    k: int = 10,
) -> float:
    """
    Calcula el Overlap@10 promedio sobre las 3 referencias.
    """
    scores = []
    print(f"{'Ref':<8} {'Oracle set':>12} {'Tus hits':>10} {'Overlap@10':>12}")
    print("-" * 46)

    for ref_id, ref_idx, ref_emb in zip(
        REFERENCE_IDS, reference_indices, reference_embs
    ):
        # Oracle: top-10 usando captions reales
        oracle_set = compute_oracle_top10(
            ref_idx, image_index, corpus_captions, corpus_ids, k=k
        )

        # Tus resultados
        my_indices, _ = search_by_reference(
            my_queries[ref_id], ref_emb.unsqueeze(0),
            image_index, clip_model, clip_processor, device,
            top_k=k, alpha=alpha
        )
        my_ids = {corpus_ids[i] for i in my_indices.tolist()}

        overlap = len(my_ids & oracle_set) / k
        scores.append(overlap)
        print(f"{ref_id:<8} {len(oracle_set):>12} {len(my_ids & oracle_set):>10} {overlap:>12.3f}")

    mean_overlap = np.mean(scores)
    print("-" * 46)
    print(f"{'Media':<8} {'':>12} {'':>10} {mean_overlap:>12.3f}  ← score del leaderboard")
    return mean_overlap


# ── Ejecutar evaluación local ──────────────────────────────────────────────────
# Cambia alpha y re-ejecuta para explorar
# score = evaluate_local(
#     my_queries, REFERENCE_INDICES, reference_embs,
#     image_index, corpus_ids, corpus_captions,
#     alpha=0.7
# )

---
## 6 — TODO C: Mejora tu sistema *(extra)*

El reto base usa un solo query de texto por imagen. Aquí puedes ir más lejos.

Algunas ideas — implementa al menos una:

**C1 — Query ensemble:** usa múltiples queries por imagen de referencia y promedia sus embeddings antes de buscar. Un ensemble de descripciones cubre más aspectos de la imagen.

```python
# Ejemplo: en lugar de un query, usa tres
queries_ref0 = [
    "soccer player kicking ball",
    "football match on grass field",
    "athlete in white jersey during sport game",
]
embs = get_text_embeddings(queries_ref0, ...)
mean_emb = F.normalize(embs.mean(dim=0, keepdim=True), p=2, dim=-1)
```

**C2 — Reciprocal Rank Fusion:** en lugar de combinar scores, combina rankings (más robusto ante diferencias de escala entre modalidades).

$$\text{RRF}(i) = \frac{1}{60 + r_{\text{texto}}(i)} + \frac{1}{60 + r_{\text{imagen}}(i)}$$

**C3 — Alpha por referencia:** en lugar de un alpha global, usa un alpha distinto para cada imagen de referencia según qué tan bien se describe con texto.

In [ ]:
# TODO C — implementa tu mejora aquí
# Compara su Overlap@10 con el TODO B usando evaluate_local()

# Tu implementación:

---
## 7 — Generar submission

El CSV tiene exactamente **10 filas por imagen de referencia** (30 filas en total).

```
ref_id,image_id,query_text
ref_0,123456,soccer player kicking ball on grass
ref_0,234567,soccer player kicking ball on grass
...   (10 filas para ref_0)
ref_1,345678,bmx cyclist jumping in the air
...
```

Incluimos `query_text` en el CSV para que el leaderboard registre qué query usaste — parte de la transparencia del experimento.

In [ ]:
def generate_submission(
    my_queries: dict,
    reference_indices: list,
    reference_embs: torch.Tensor,
    image_index: torch.Tensor,
    corpus_ids: list,
    output_path: str = "submission.csv",
    alpha: float = 0.7,
    top_k: int = 10,
) -> pd.DataFrame:
    rows = []
    for ref_id, ref_emb in zip(REFERENCE_IDS, reference_embs):
        indices, _ = search_by_reference(
            my_queries[ref_id], ref_emb.unsqueeze(0),
            image_index, clip_model, clip_processor, device,
            top_k=top_k, alpha=alpha
        )
        for idx in indices.tolist():
            rows.append({
                "ref_id":     ref_id,
                "image_id":   corpus_ids[idx],
                "query_text": my_queries[ref_id],
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)

    print(f"✅ Submission guardada → '{output_path}'")
    print(f"   {len(df)} filas | {df['ref_id'].nunique()} referencias | top-{top_k} cada una")
    display(df)
    return df


# ── Generar (descomenta cuando estés listo) ────────────────────────────────────
# df_sub = generate_submission(
#     my_queries, REFERENCE_INDICES, reference_embs,
#     image_index, corpus_ids,
#     alpha=0.7,   # ← tu mejor alpha
# )
#
# from google.colab import files
# files.download("submission.csv")

---
## 8 — Reflexión final

**1. Queries**  
¿Qué estrategia usaste para escribir los queries? ¿Priorizaste la acción, el entorno, los objetos, el estilo?

*Tu respuesta:*

---

**2. Alpha**  
¿Usaste el mismo alpha para las 3 referencias o uno distinto por cada una? ¿Por qué?

*Tu respuesta:*

---

**3. Mejora (TODO C)**  
¿Qué mejora implementaste y cuánto subió el Overlap@10?

*Tu respuesta:*

---

**4. Limitaciones**  
¿En qué casos falla tu sistema? ¿Qué harías diferente con más tiempo?

*Tu respuesta:*